In [ ]:
import torch
import os
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
import huggingface_hub

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt 

from timm import create_model  # or torchvision.models
from tqdm.notebook import tqdm
from torchinfo import summary
from pipeline_func import set_seed, build_dataset, get_lr_lambda, decode_signal

In [2]:

# Automatically detect the best device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [3]:
timm.list_models("efficientnetv*")

['efficientnetv2_l',
 'efficientnetv2_m',
 'efficientnetv2_rw_m',
 'efficientnetv2_rw_s',
 'efficientnetv2_rw_t',
 'efficientnetv2_s',
 'efficientnetv2_xl']

In [2]:
class CFG:
    verbose = 1  # Verbosity
    seed = 42  # Random seed
    preset = "efficientnetv2_l"  # Name of pretrained classifier
    image_size = [400, 300]  # Input image size
    epochs = 13 # Training epochs
    batch_size = 8  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    drop_remainder = True  # Drop incomplete batches
    num_classes = 6 # Number of classes in the dataset
    fold = 0 # Which fold to set as validation data
    class_names = ['Seizure', 'LPD', 'GPD', 'LRDA','GRDA', 'Other']
    label2name = dict(enumerate(class_names))
    name2label = {v:k for k, v in label2name.items()}
    output_dir = "./effnet_result"

In [3]:
CFG.label2name

{0: 'Seizure', 1: 'LPD', 2: 'GPD', 3: 'LRDA', 4: 'GRDA', 5: 'Other'}

In [6]:
BASE_PATH = "./Data"
SPEC_DIR = "./tmp/kaggle_spectrogram"
os.makedirs(SPEC_DIR+'/train_spectrograms', exist_ok=True)
# os.makedirs(SPEC_DIR+'/test_spectrograms', exist_ok=True)

In [7]:
# Train + Valid
df = pd.read_csv(f'{BASE_PATH}/train.csv')
df['eeg_path'] = f'{BASE_PATH}/train_eegs/'+df['eeg_id'].astype(str)+'.parquet'
df['spec_path'] = f'{BASE_PATH}/train_spectrograms/'+df['spectrogram_id'].astype(str)+'.parquet'
df['spec_path2'] = f'{SPEC_DIR}/train_spectrograms/'+df['spectrogram_id'].astype(str)+'.npy'
df['class_name'] = df.expert_consensus.copy()
df['class_label'] = df.expert_consensus.map(CFG.name2label)
display(df.head(10))

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,eeg_path,spec_path,spec_path2,class_name,class_label
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
5,1628180742,5,26.0,353733,5,26.0,2413091605,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
6,1628180742,6,30.0,353733,6,30.0,364593930,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
7,1628180742,7,36.0,353733,7,36.0,3811483573,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
8,1628180742,8,40.0,353733,8,40.0,3388718494,42516,Seizure,3,0,0,0,0,0,./Data/train_eegs/1628180742.parquet,./Data/train_spectrograms/353733.parquet,./tmp/kaggle_spectrogram/train_spectrograms/35...,Seizure,0
9,2277392603,0,0.0,924234,0,0.0,1978807404,30539,GPD,0,0,5,0,1,5,./Data/train_eegs/2277392603.parquet,./Data/train_spectrograms/924234.parquet,./tmp/kaggle_spectrogram/train_spectrograms/92...,GPD,2


In [8]:
# Define a function to process a single eeg_id
def process_spec(spec_id, split="train"):
    spec_path = f"{BASE_PATH}/{split}_spectrograms/{spec_id}.parquet"
    spec = pd.read_parquet(spec_path)
    spec = spec.fillna(0).values[:, 1:].T # fill NaN values with 0, transpose for (Time, Freq) -> (Freq, Time)
    spec = spec.astype("float32")
    np.save(f"{SPEC_DIR}/{split}_spectrograms/{spec_id}.npy", spec)

# Get unique spec_ids of train and valid data
spec_ids = df["spectrogram_id"].unique()
subset_size = int(0.2 * len(spec_ids))
spec_ids_subset = np.random.choice(spec_ids, size=subset_size, replace=False)
sub_df = df[df["spectrogram_id"].isin(spec_ids_subset)]

# Parallelize the processing using joblib for training data
"""_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(process_spec)(spec_id, "train")
    for spec_id in tqdm(spec_ids_subset, total=len(spec_ids_subset))
)
"""

'_ = joblib.Parallel(n_jobs=-1, backend="loky")(\n    joblib.delayed(process_spec)(spec_id, "train")\n    for spec_id in tqdm(spec_ids_subset, total=len(spec_ids_subset))\n)\n'

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)

sub_df["fold"] = -1
sub_df.reset_index(drop=True, inplace=True)
for fold, (train_idx, valid_idx) in enumerate(
    sgkf.split(sub_df, y=sub_df["class_label"], groups=sub_df["patient_id"])
):
    sub_df.loc[valid_idx, "fold"] = fold
sub_df.groupby(["fold", "class_name"])[["eeg_id"]].count()

/var/folders/v1/3rps7g510r30px4lkmvj73kr0000gq/T/ipykernel_57846/3167152088.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_df["fold"] = -1


eeg_id
fold class_name        
0    GPD           1757
     GRDA          1159
     LPD            357
     LRDA           667
     Other          830
     Seizure        649
1    GPD            280
     GRDA           429
     LPD            235
     LRDA           436
     Other          635
     Seizure        758
2    GPD           1073
     GRDA           521
     LPD            790
     LRDA           634
     Other          754
     Seizure        842
3    GPD            672
     GRDA           485
     LPD            827
     LRDA           449
     Other          790
     Seizure        819
4    GPD            474
     GRDA           561
     LPD           1329
     LRDA           613
     Other          954
     Seizure        704

In [12]:
# Build Train & Valid Dataset¶

# Sample from full data
sample_df = sub_df.groupby("spectrogram_id").head(1).reset_index(drop=True)
train_df = sample_df[sample_df.fold != CFG.fold]
valid_df = sample_df[sample_df.fold == CFG.fold]
print(f"# Num Train: {len(train_df)} | Num Valid: {len(valid_df)}")

# Num Train: 1792 | Num Valid: 435


In [13]:

# Train
train_paths = train_df.spec_path2.values
train_offsets = train_df.spectrogram_label_offset_seconds.values.astype(int)
train_labels = train_df.class_label.values
train_ds = build_dataset(train_paths, train_offsets, train_labels, batch_size=CFG.batch_size, decode_fn=decode_signal,
                         repeat=False, shuffle=True, augment=True, cache=True)

# Valid
valid_paths = valid_df.spec_path2.values
valid_offsets = valid_df.spectrogram_label_offset_seconds.values.astype(int)
valid_labels = valid_df.class_label.values
valid_ds = build_dataset(valid_paths, valid_offsets, valid_labels, batch_size=CFG.batch_size, decode_fn=decode_signal,
                         repeat=False, shuffle=False, augment=False, cache=True)

In [14]:
"""def decode_fn(path):
    # Load and return image from .npy file
    img = np.load(path)
    return img

imgs, tars = next(iter(train_ds))  # imgs is list/tensor of file paths

num_imgs = 8
plt.figure(figsize=(4*4, (num_imgs // 4 + (num_imgs % 4 > 0)) * 4))

for i in range(num_imgs):
    plt.subplot(num_imgs // 4 + (num_imgs % 4 > 0), 4, i + 1)

    # Decode path string if it's a tensor
    path = imgs[i]

    img = path[0,:,:]

    # If image has shape [H, W, 1], squeeze it
    if img.ndim == 3 and img.shape[-1] == 1:
        img = img[..., 0]

    # Normalize for display
    img = (img - img.min()) / (img.max() - img.min() + 1e-5)

    # Get label
    tar = CFG.label2name[int(tars[i])] if hasattr(CFG, "label2name") else str(int(tars[i]))

    plt.imshow(img, cmap='jet')
    plt.title(f"Target: {tar}")
    # plt.axis('off')

plt.tight_layout()
plt.show()"""

'def decode_fn(path):\n    # Load and return image from .npy file\n    img = np.load(path)\n    return img\n\nimgs, tars = next(iter(train_ds))  # imgs is list/tensor of file paths\n\nnum_imgs = 8\nplt.figure(figsize=(4*4, (num_imgs // 4 + (num_imgs % 4 > 0)) * 4))\n\nfor i in range(num_imgs):\n    plt.subplot(num_imgs // 4 + (num_imgs % 4 > 0), 4, i + 1)\n\n    # Decode path string if it\'s a tensor\n    path = imgs[i]\n\n    img = path[0,:,:]\n\n    # If image has shape [H, W, 1], squeeze it\n    if img.ndim == 3 and img.shape[-1] == 1:\n        img = img[..., 0]\n\n    # Normalize for display\n    img = (img - img.min()) / (img.max() - img.min() + 1e-5)\n\n    # Get label\n    tar = CFG.label2name[int(tars[i])] if hasattr(CFG, "label2name") else str(int(tars[i]))\n\n    plt.imshow(img, cmap=\'jet\')\n    plt.title(f"Target: {tar}")\n    # plt.axis(\'off\')\n\nplt.tight_layout()\nplt.show()'

In [15]:
# Define KL Divergence loss (note: log_softmax + softmax expected)
criterion = nn.KLDivLoss(reduction='batchmean')

# Build the classifier from a preset (e.g., timm model)
model = create_model(CFG.preset, pretrained=False, num_classes=CFG.num_classes)

# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# learning rate scheduler
lr_lambda = get_lr_lambda(batch_size=CFG.batch_size, mode='cos', epochs=CFG.epochs)

# Move model to device
model = model.to(device)

In [16]:
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val_loss = float('inf')
os.makedirs(CFG.output_dir, exist_ok=True)

history = {'train_loss': [], 'val_loss': []}


In [ ]:
for epoch in range(CFG.epochs):
# for epoch in range(1):
        model.train()
        train_loss = 0
        for batch in tqdm(train_ds, desc=f"Epoch {epoch+1}/{CFG.epochs}", disable=CFG.verbose==0):
            inputs, targets = batch
            inputs, targets = inputs.to(device), targets.to(device)
            targets_one_hot = F.one_hot(targets, num_classes=CFG.num_classes).float()

            optimizer.zero_grad()
            outputs = model(inputs)
            # Convert outputs to log-probabilities
            log_probs = F.log_softmax(outputs, dim=1)
            loss = criterion(log_probs, targets_one_hot)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)

        scheduler.step()
        avg_train_loss = train_loss / len(train_ds.dataset)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in valid_ds:
                inputs, targets = batch
                inputs, targets = inputs.to(device), targets.to(device)
                targets_one_hot = F.one_hot(targets, num_classes=CFG.num_classes).float()

                outputs = model(inputs)
                log_probs = F.log_softmax(outputs, dim=1)

                loss = criterion(log_probs, targets_one_hot)
                val_loss += loss.item() * inputs.size(0)

        avg_val_loss = val_loss / len(valid_ds.dataset)

        if CFG.verbose > 0:
            print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}")

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(CFG.output_dir, 'best_model.pt'))


Epoch 1/13:   0%|          | 0/224 [00:00<?, ?it/s]

Epoch 1: train_loss=2.7125, val_loss=14.8189


Epoch 2/13:   0%|          | 0/224 [00:00<?, ?it/s]

Epoch 2: train_loss=1.7723, val_loss=122.4146


Epoch 3/13:   0%|          | 0/224 [00:00<?, ?it/s]

...

Epoch 11: train_loss=1.1590, val_loss=18.5597


Epoch 12/13:   0%|          | 0/224 [00:00<?, ?it/s]

Epoch 12: train_loss=1.0662, val_loss=34.3231


Epoch 13/13:   0%|          | 0/224 [00:00<?, ?it/s]

Epoch 13: train_loss=0.9908, val_loss=12.1036


In [19]:

# Print model summary (optional)
summary(model, input_size=(1, 3, 400, 380))

Layer (type:depth-idx)                        Output Shape              Param #
EfficientNet                                  [1, 6]                    --
├─Conv2d: 1-1                                 [1, 32, 200, 190]         864
├─BatchNormAct2d: 1-2                         [1, 32, 200, 190]         64
│    └─Identity: 2-1                          [1, 32, 200, 190]         --
│    └─SiLU: 2-2                              [1, 32, 200, 190]         --
├─Sequential: 1-3                             [1, 640, 13, 12]          --
│    └─Sequential: 2-3                        [1, 32, 200, 190]         --
│    │    └─ConvBnAct: 3-1                    [1, 32, 200, 190]         9,280
│    │    └─ConvBnAct: 3-2                    [1, 32, 200, 190]         9,280
│    │    └─ConvBnAct: 3-3                    [1, 32, 200, 190]         9,280
│    │    └─ConvBnAct: 3-4                    [1, 32, 200, 190]         9,280
│    └─Sequential: 2-4                        [1, 64, 100, 95]          --
│    │ 